In [1]:
!pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from torch import fx  

import os
import json
from tqdm import tqdm

import bitsandbytes as bnb

# Load Hugging Face and Quantization libraries AFTER core torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
INPUT_FILE = "/kaggle/input/datasets/sayonsujitmondal/gold-prompt/gold_prompts_wiki.jsonl"
OUTPUT_FILE = "/kaggle/working/gold_only_model_results.jsonl"

print("bitsandbytes version:", bnb.__version__)
print(torch.version.cuda)

bitsandbytes version: 0.49.2
12.8


In [3]:
# Verify Input File
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"CRITICAL ERROR: Input file not found at {INPUT_FILE}. "
                            "Please check your dataset slug in the right-hand sidebar.")
else:
    print(f"✓ Input file found: {INPUT_FILE}")

# Verify Output Permissions
try:
    with open(OUTPUT_FILE, 'a'): pass
    print(f"✓ Output directory writable: {OUTPUT_FILE}")
except Exception as e:
    raise PermissionError(f"CRITICAL ERROR: Cannot write to output path. {e}")

✓ Input file found: /kaggle/input/datasets/sayonsujitmondal/gold-prompt/gold_prompts_wiki.jsonl
✓ Output directory writable: /kaggle/working/gold_only_model_results.jsonl


In [4]:
# Verify Hugging Face Authentication
user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("✓ Hugging Face authentication successful.")
    else:
        raise ValueError("HF_TOKEN not found in Kaggle Secrets!")
except Exception as e:
    raise RuntimeError(f"CRITICAL ERROR: Auth failed. {e}")

✓ Hugging Face authentication successful.


In [5]:
# Verify GPU Availability
if not torch.cuda.is_available():
    print("WARNING: GPU not detected. Model loading will likely fail or be extremely slow.")
else:
    print(f"✓ GPU Detected: {torch.cuda.get_device_name(0)}")

print("--- All checks passed. Proceeding to Model Load ---")


✓ GPU Detected: Tesla T4
--- All checks passed. Proceeding to Model Load ---


In [6]:
# 3. Load Model with 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Loading {MODEL_NAME}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"": 0},
        trust_remote_code=True
    )
    print("✓ Model and Tokenizer loaded successfully!")
except Exception as e:
    print(f"ERROR: Model load failed.  Details: {e}")
    raise


Loading meta-llama/Llama-2-7b-chat-hf...


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

✓ Model and Tokenizer loaded successfully!


In [8]:
# Inference Function
def generate_answer(prompt):
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        # Greedy generation (do_sample=False) with 15 token max
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=15, 
                do_sample=False, 
                pad_token_id=tokenizer.eos_token_id
            )
        
        full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the newly generated text
        return full_text[len(prompt):].strip()
    except Exception as e:
        return f"ERROR_IN_INFERENCE: {str(e)}"

In [9]:
# Execution Loop
LIMIT = 300  # Set your desired number of questions here

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    all_lines = f.readlines()

lines = all_lines[:LIMIT]
total_samples = len(lines)
print(f"Starting inference on {total_samples} samples (Limited from {len(all_lines)})...")

# Track progress with a counter
processed_count = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:
    for line in tqdm(lines, desc="Total Progress"):
        try:
            data = json.loads(line)
            prompt_text = data["prompt"]
            
            prediction = generate_answer(prompt_text)
            
            result_entry = {
                "id": data["id"],
                "gold_answer": data["answer"],
                "model_output": prediction
            }
            
            out_f.write(json.dumps(result_entry, ensure_ascii=False) + "\n")
            
            processed_count += 1
            
            # Print explicit progress every 50 samples
            if processed_count % 50 == 0:
                print(f"--- Progress Update: Processed {processed_count}/{total_samples} samples ---")
                out_f.flush()
                
        except Exception as e:
            print(f"Skipping error at index {processed_count}: {e}")
            continue

print(f"Inference complete! {processed_count} results saved to: {OUTPUT_FILE}")

Starting inference on 300 samples (Limited from 1200)...



Total Progress:   0%|          | 0/300 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

Total Progress:  17%|█▋        | 50/300 [00:39<03:16,  1.27it/s]

--- Progress Update: Processed 50/300 samples ---



Total Progress:  33%|███▎      | 100/300 [01:15<02:03,  1.62it/s]

--- Progress Update: Processed 100/300 samples ---


...

--- Progress Update: Processed 200/300 samples ---



Total Progress:  83%|████████▎ | 250/300 [03:24<00:32,  1.53it/s]

--- Progress Update: Processed 250/300 samples ---



Total Progress: 100%|██████████| 300/300 [04:10<00:00,  1.20it/s]

--- Progress Update: Processed 300/300 samples ---
Inference complete! 300 results saved to: /kaggle/working/gold_only_model_results.jsonl
